# CV Masterclass 14: Human Pose Estimation & Keypoints

Welcome to Notebook 14. In previous modules, we focused on 'what' is in an image (Classification) and 'where' it is (Detection/Segmentation). Today, we move into **Structural Understanding**. 

Human Pose Estimation (HPE) is the task of localizing specific anatomical 'keypoints' (joints) and modeling their spatial relationships (the skeleton). This is the foundation for action recognition, AR/VR, and human-computer interaction.

---

## 🎓 Socratic Deep Check
Ponder these questions before we begin:

> *"In heatmap regression, we typically use a Gaussian kernel with a fixed $\sigma=2$. If a person is very far away (small), the heatmap peak becomes relatively massive compared to the joint size. Why does this fixed $\sigma$ induce 'quantization error' at lower resolutions, and how do coordinate-refined methods mathematically compensate for the sub-pixel shift?"*

> *"OpenPose uses Part Affinity Fields (PAFs) to perform bipartite matching. If two people are standing close, the left arm of Person A might be physically closer to the torso of Person B than to its own. Why is a simple Euclidean distance matcher insufficient, and how do the internal vector dot products of PAFs provide the 'structural rigidity' needed to solve the grouping problem?"*

---

## Table of Contents
1. **Top-Down vs Bottom-Up Paradigms:** Accuracy vs Multi-person latency.
2. **Heatmap Math:** Gaussian regression, $\text{argmax}$ non-differentiability, and **Soft-argmax**.
3. **HRNet Architecture:** Maintaining High-Resolution representations concurrently.
4. **OpenPose & PAFs:** Solving the Bipartite Matching problem with Vector Fields.
5. **Evaluation:** Object Keypoint Similarity (OKS).


## 1. Top-Down vs Bottom-Up Paradigms

In multi-person pose estimation, we face a fundamental choice in pipeline geometry.

### Top-Down (Detect-then-Pose)
1.  **Step 1:** Run an Object Detector (like YOLO or Faster R-CNN) to find all person bounding boxes.
2.  **Step 2:** Crop each person and resize to a fixed input (e.g., $256 \times 192$).
3.  **Step 3:** Run a single-person pose estimator on each crop.

**Pros:** Extremely high accuracy (normalized scale). 
**Cons:** Latency scales linearly $O(N)$ with the number of people. If $N=50$ (a marathon), the system grinds to a halt.

### Bottom-Up (Detect-then-Group)
1.  **Step 1:** Detect *all* joints in the image simultaneously (e.g., find all 100 elbows).
2.  **Step 2:** Group/Associate these joints into individual skeletons.

**Pros:** Constant latency $O(1)$ regardless of person count. 
**Cons:** Lower accuracy on small/overlapping people; association is a hard combinatorial problem.

| Metric | Top-Down (e.g., AlphaPose, HRNet) | Bottom-Up (e.g., OpenPose, HigherHRNet) |
|--------|-----------------------------------|-----------------------------------------|
| Accuracy | 66666 (Higher) | 6666 (Lower) |
| Complexity | $O(\text{Persons})$ | $O(\text{Image Size})$ |
| Robustness to Occlusion | High (local context) | Low (global association errors) |

## 2. Localization Math: Heatmaps vs. Coordinate Regression

How do we represent a joint location $(x, y)$ in a neural network?

### Direct Coordinate Regression
Predicting $x, y \in [0, 1]$ directly via a Fully Connected (FC) layer.
-   **Problem:** Spatial information is collapsed into a single vector. FC layers lack 'spatial bias', making it hard for the model to learn the 'look' of an elbow across different translations.

### Heatmap Regression (The Gold Standard)
Predicting a 2D image $H$ where the value at each pixel $(u, v)$ represents the probability of the joint being there.
We supervise using a **2D Gaussian Heatmap** centered at ground-truth $(x, y)$:
$$ G(u, v) = \exp\left( -\frac{(u-x)^2 + (v-y)^2}{2\sigma^2} \right) $$

#### The Problem: Non-Differentiable Argmax
During inference, we want $(x, y) = \text{argmax}(H)$. However, $\text{argmax}$ is a jump function; its derivative is zero everywhere it is defined. We cannot backpropagate through it.

#### The Solution: Spatial Softmax (Soft-argmax)
We treat the heatmap as a probability distribution and compute the **Expected Value** of the coordinates:
1.  **Softmax:** $P(u, v) = \frac{\exp(H(u,v))}{\sum_{i,j} \exp(H(i,j))}$
2.  **Expectation:** $\hat{x} = \sum_{u,v} u \cdot P(u,v), \quad \hat{y} = \sum_{u,v} v \cdot P(u,v)$

This is fully differentiable and allows for **sub-pixel** accuracy!

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# -----------------------------------------------------
# IMPLEMENTATION: 2D Gaussian Heatmap Generator
# -----------------------------------------------------

def generate_gaussian_heatmap(size, center, sigma=2.0):
    """
    Generates a 2D Gaussian heatmap centered at 'center'.
    size: (H, W)
    center: (x, y)
    """
    H, W = size
    x0, y0 = center
    
    # Create coordinate grids
    yy, xx = torch.meshgrid(torch.arange(H).float(), torch.arange(W).float(), indexing='ij')
    
    # Compute Gaussian
    heatmap = torch.exp(-((xx - x0)**2 + (yy - y0)**2) / (2 * sigma**2))
    return heatmap

class SoftArgmax2D(nn.Module):
    """
    Converts a 2D heatmap to (x, y) coordinates differentiably.
    """
    def __init__(self, beta=100.0):
        super().__init__()
        self.beta = beta # Temperature to sharpen the distribution

    def forward(self, x):
        # x: [B, K, H, W] - B: batch, K: keypoints
        B, K, H, W = x.shape
        
        # 1. Flatten and Softmax
        # x: [B, K, H*W]
        probs = F.softmax(x.view(B, K, -1) * self.beta, dim=-1)
        probs = probs.view(B, K, H, W)
        
        # 2. Create Coordinate Grids
        # Grid relative to [0, 1] for normalization
        yy, xx = torch.meshgrid(torch.linspace(0, 1, H), torch.linspace(0, 1, W), indexing='ij')
        yy = yy.to(x.device)
        xx = xx.to(x.device)
        
        # 3. Weighted Sum (Expectation)
        # probs: [B, K, H, W]
        expected_x = torch.sum(probs * xx, dim=(2, 3))
        expected_y = torch.sum(probs * yy, dim=(2, 3))
        
        return torch.stack([expected_x, expected_y], dim=-1)

# TEST IT
H, W = 64, 64
target_center = (42.5, 12.3) # Sub-pixel target
gt_heatmap = generate_gaussian_heatmap((H, W), target_center)

soft_argmax = SoftArgmax2D()
# Input needs batch/keypoint dimensions: [1, 1, 64, 64]
pred_coords = soft_argmax(gt_heatmap.unsqueeze(0).unsqueeze(0))

# Convert normalized [0, 1] back to pixel scale
pred_x = pred_coords[0, 0, 0].item() * (W - 1)
pred_y = pred_coords[0, 0, 1].item() * (H - 1)

print(f"Target: {target_center}")
print(f"Soft-argmax Prediction: ({pred_x:.2f}, {pred_y:.2f})")
assert abs(pred_x - target_center[0]) < 0.1
assert abs(pred_y - target_center[1]) < 0.1

plt.imshow(gt_heatmap.numpy(), cmap='hot')
plt.title(f"Gaussian Heatmap centered at {target_center}")
plt.scatter([target_center[0]], [target_center[1]], color='blue', marker='x', label='Ground Truth')
plt.legend()
plt.show()

### COMMON PITFALLS
*   **Heatmap Quantization:** If your heatmap resolution is small (e.g., $1/8$ of input), a single pixel error in the heatmap results in an 8-pixel error in the high-res image. This is why **HRNet** is so critical—it avoids downsampling too much.
*   **The Edge Case:** If a joint is occluded, the network might predict a 'flat' heatmap. `argmax` will pick a random noise peak, whereas `soft-argmax` will average the noise and predict the center of the image. You must check the heatmap *maximum value* as a confidence score!

## 3. High-Resolution Net (HRNet)

Traditional architectures (ResNet, Stacked Hourglass) follow a **Series** connection: High-Res → Low-Res → High-Res.

**The Problem:** The signal goes through a bottleneck. When we downsample to $1/32$, we lose fine spatial details of fingers or facial features that are impossible to recover perfectly during upsampling.

**The HRNet Innovation:**
HRNet maintains a **High-Resolution representation** throughout the entire network. New branches are added in **Parallel** instead of sequence.

1.  **Parallel Multi-Resolution:** High, Medium, and Low resolutions run concurrently.
2.  **Repeated Multi-Scale Fusion:** Information is constantly exchanged between branches via convolutions (downsampling) and transposed convolutions (upsampling).

This ensures that the final high-resolution feature map posesses 'semantic richness' from the low-res paths while preserving 'spatial precision' from the high-res paths.

In [ ]:
class HRNetFusionBlock(nn.Module):
    """
    Conceptual implementation of HRNet Multi-Scale Fusion.
    Combines a High-Res stream and a Low-Res stream.
    """
    def __init__(self, high_dim=32, low_dim=64):
        super().__init__()
        # Downsample High -> Low
        self.down = nn.Conv2d(high_dim, low_dim, kernel_size=3, stride=2, padding=1)
        # Upsample Low -> High
        self.up = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(low_dim, high_dim, kernel_size=1)
        )
        
    def forward(self, x_high, x_low):
        # Cross-fusion
        new_high = x_high + self.up(x_low)
        new_low = x_low + self.down(x_high)
        return new_high, new_low

# TEST IT
fusion = HRNetFusionBlock()
h_in = torch.randn(1, 32, 64, 64)
l_in = torch.randn(1, 64, 32, 32)

h_out, l_out = fusion(h_in, l_in)
assert h_out.shape == h_in.shape
assert l_out.shape == l_in.shape
print(f"Fused High-Res Branch: {h_out.shape}")
print(f"Fused Low-Res Branch: {l_out.shape}")
print("Information has been exchanged between both resolutions concurrently!")

## 4. OpenPose & Part Affinity Fields (PAFs)

For **Bottom-Up** multi-person pose, detection is easy; association is hard.
How do we know which *elbow* belongs to which *shoulder* when people are overlapping?

**The Old Way (Greedy/Pairwise):** Match every elbow to the nearest shoulder. Fail! (People overlap).

**The OpenPose Way (Part Affinity Fields):**
Instead of just points, OpenPose predicts **vector fields** (PAFs) along the limbs. 
A PAF $L_{k}(p)$ is a 2D vector for every pixel $p$ on a limb, pointing from Part A to Part B.

### Math: The Line Integral
To score the connection between a detected shoulder ($j_1$) and an elbow ($j_2$), we compute the **Line Integral** of the PAF along the candidate limb segment:
$$ E = \int_{u=0}^{u=1} L(p(u)) \cdot \frac{j_2 - j_1}{||j_2 - j_1||} du $$

Where $L(p(u))$ is the predicted vector field at point $u$ on the segment. 
- If $L$ aligns perfectly with the direction of the segment, the dot product is 1, and the score is high.
- If $L$ points elsewhere, the score is low.

We then solve the **Bipartite Matching** problem (optimal assignment) using the Hungarian Algorithm or a greedy approach based on these scores.

In [ ]:
# -----------------------------------------------------
# IMPLEMENTATION: PAF-based Limb Scoring
# -----------------------------------------------------

def score_limb_connection(paf_field, j1, j2, num_samples=10):
    """
    paf_field: [2, H, W] tensor (Ux, Uy vectors)
    j1, j2: (x, y) coordinates of two joints
    """
    # 1. Vector from j1 to j2
    limb_vec = torch.tensor([j2[0] - j1[0], j2[1] - j1[1]], dtype=torch.float32)
    norm = torch.norm(limb_vec)
    if norm == 0: return 0.0
    limb_dir = limb_vec / norm
    
    # 2. Sample points along the segment
    scores = []
    for i in range(num_samples):
        # Interpolate location
        u = i / (num_samples - 1)
        curr_x = int(j1[0] + u * limb_vec[0])
        curr_y = int(j1[1] + u * limb_vec[1])
        
        # 3. Dot product with predicted PAF
        paf_vec = paf_field[:, curr_y, curr_x]
        dot_prod = torch.dot(paf_vec, limb_dir)
        scores.append(dot_prod)
        
    return sum(scores) / len(scores)

# TEST IT
H, W = 100, 100
paf = torch.zeros(2, H, W)
paf[0, 20:80, 50] = 1.0 # Vectors pointing straight down (+y direction)

j_shoulder = (50, 20)
j_elbow_good = (50, 80)
j_elbow_bad = (80, 80)

score_good = score_limb_connection(paf, j_shoulder, j_elbow_good)
score_bad = score_limb_connection(paf, j_shoulder, j_elbow_bad)

print(f"Correct connection score: {score_good:.4f}")
print(f"Incorrect connection score: {score_bad:.4f}")
assert score_good > score_bad


## 5. Evaluation: Object Keypoint Similarity (OKS)

Standard mAP uses Intersection over Union (IoU). But keypoints have zero area! We need a distance-based equivalent. 

**OKS Formula:**
$$ OKS = \frac{\sum_i \exp(-d_i^2 / 2s^2 k_i^2) \delta(v_i > 0)}{\sum_i \delta(v_i > 0)} $$

Where:
- $d_i$: Euclidean distance between predicted and GT keypoint $i$.
- $s$: Scale of the person (area of bounding box).
- $k_i$: Per-keypoint constant (standard deviation of human annotator error). Head keypoints have lower $k$ (high precision needed) than hips.
- $v_i$: Visibility flag.

**Why use exponential decay?**
It treats 'near misses' gracefully. A 2-pixel error on a massive person should be penalized less than a 2-pixel error on a tiny person. OKS handles this **scale normalization** automatically.

In [ ]:
def compute_oks(dists, scale, k_values):
    """
    Simplified OKS calculation.
    dists: [K] Euclidean distances
    scale: Scalar (sqrt of person area)
    k_values: [K] Constants
    """
    vars = (scale * k_values) ** 2
    oks_vals = torch.exp(-dists**2 / (2 * vars))
    return oks_vals.mean()

# TEST IT
distances = torch.tensor([2.0, 5.0, 1.0]) # Errors in pixels
k_consts = torch.tensor([0.026, 0.072, 0.089]) # [Head, Shoulder, Knee]

large_scale = 100.0
small_scale = 20.0

oks_large = compute_oks(distances, large_scale, k_consts)
oks_small = compute_oks(distances, small_scale, k_consts)

print(f"OKS for Large Person: {oks_large:.4f}")
print(f"OKS for Small Person: {oks_small:.4f}")
assert oks_large > oks_small # Same pixel error is much worse on a small person!

### 🎓 DEEP QUESTION ANSWERED

**Q:** *Why does a simple Euclidean distance matcher fail in OpenPose association?*

**A:** 
Euclidean distance is 'blind' to structural orientation. If Person A's left elbow is physically closer to Person B's torso than to Person A's torso, a distance-based algorithm will merge them into a single mutant skeleton. 

Part Affinity Fields solve this by enforcing **Vector alignment**. Even if Person B's torso is closer, the vector field between Person A's shoulder and elbow points in a consistent direction along the limb. The line integral 'collects' this alignment evidence, ensuring that the connection follows the physical limb geometry, not just proximity.